In [1]:
import pandas as pd
import numpy as np

In [2]:
meta = pd.read_csv("meta.csv")
# df = pd.read_csv("example.csv")

df = pd.read_clipboard()
# df.to_csv("example.csv", index=False)
df.head(10)

,826,MB,EF,GNL,WT
0,0,0,0,0,1
1,0,0,0,0,1
2,0,0,0,0,1
3,0,0,0,0,1
4,0,0,0,0,1
5,0,0,0,0,1
6,0,5,1,0,1
7,1,3,1,0,1
8,1,1,5,0,1
9,1,3,1,1,1


In [3]:
partners = df.columns[:-1].tolist()

# wieghts model
Wp = df[partners].to_numpy()

# category weights (current unused)
# Wc = np.ones(Wp.shape[0])
Wc = df[df.columns[-1]].to_numpy()
partners, Wp.shape, Wc.shape

(['826', 'MB', 'EF', 'GNL'], (39, 4), (39,))

In [4]:

# make the rules based on the spreadsheet
rule_text = [
    "Willing to have direct contact with kids",
    "Wants to work with HS",
    "Available in-person",
    "Weekly avail required",
    "Weekday avail required",
]

rules = np.zeros((5, len(Wp)))
(contact, hs, ip, weekly, weekday) = range(5)

rules[contact][[19, 20]] = 1
rules[hs][18] = 1
rules[ip][[23, 24]] = 1
rules[weekly][30] = 1
rules[weekday][[23, 24, 25, 26, 28, 29]] = 1
rules[contact]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.])

In [5]:
# fake a person
x = np.random.randint(0, 2, Wp.shape[0])
x

array([1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0,
       1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1], dtype=int32)

In [6]:
# initial recommendation
recommendation = Wc * x @ Wp
recommendation, partners[np.argmax(recommendation)]

(array([18, 17, 15, 21]), 'GNL')

In [ ]:
passed = rules @ x > 0.5
# GNL never gets removed
Er = np.array(
    [
        passed[contact] and passed[ip],
        passed[contact] and passed[hs],
        passed[ip] and passed[weekly] and passed[weekday],
        True,
    ],
    dtype=np.int64,
)

with_rules = recommendation * Er

print(" person:", ",".join([s for i, s in list(zip(x, meta["Symbol"])) if i > 0.5]))
print("initial:", partners[np.argmax(recommendation)], recommendation)
print("w rules:", partners[np.argmax(with_rules)], with_rules)
print(" passed:", passed)
print("     Er:", Er)

for str, p in zip(rule_text, passed):
    if not p:
        print(" * req:", str)

 person: general_support,local_school,alumni,alumni_parent,career,event,gardening,remote_friendly,other,elementary,middle,direct_1to1,other,weekday_morning_remote,weekday_afternoon_remote,weekday_evening,unsure,name,phone
initial: GNL [18 17 15 21]
w rules: GNL [ 0  0  0 21]
 passed: [ True False False False  True]
     Er: [0 0 0 1]
 * req: Wants to work with HS
 * req: Available in-person
 * req: Weekly avail required
